<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Sterimol_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# INSTALL (run once in Google Colab)
# ============================================================
!pip install morfeus-ml pandas numpy openpyxl

# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive', force_remount=True)


# ============================================================
# IMPORTS
# ============================================================
import os
import pandas as pd

from morfeus import Sterimol, read_xyz


# ============================================================
# PATH TO XYZ FILES
# ============================================================
xyz_folder = "/content/drive/MyDrive/xyz_files"


# ============================================================
# FIXED ATOM NUMBERING IN ALL STRUCTURES
# ============================================================

C2_ATOM = 18          # C2 atom
R3_START_ATOM = 44   # First atom directly attached to C2


# ============================================================
# R3 GROUP NAMES
# Change filenames to match your actual XYZ filenames
# ============================================================

R3_names = {
    "RR_alkene.xyz": "alkene",
    "RR_alkyne.xyz": "alkyne"
}


# ============================================================
# STORAGE
# ============================================================

results = []


# ============================================================
# PROCESS XYZ FILES
# ============================================================

xyz_files = [
    f for f in os.listdir(xyz_folder)
    if f.lower().endswith(".xyz")
]


for file in xyz_files:

    path = os.path.join(xyz_folder, file)

    try:

        # ----------------------------------------------------
        # Read XYZ
        # ----------------------------------------------------
        elements, coordinates = read_xyz(path)

        n_atoms = len(elements)


        # ----------------------------------------------------
        # Check numbering
        # ----------------------------------------------------
        if n_atoms < R3_START_ATOM:

            raise ValueError(
                f"Structure has only {n_atoms} atoms, "
                f"but R3 starts at atom {R3_START_ATOM}"
            )


        # ----------------------------------------------------
        # R3 = atoms 28 through the last atom
        # ----------------------------------------------------
        R3_atoms = list(
            range(R3_START_ATOM, n_atoms + 1)
        )


        # ----------------------------------------------------
        # Exclude everything except R3
        #
        # Keep:
        #   atom 2  = C2 / dummy atom
        #   atoms 28-last = R3
        #
        # Exclude:
        #   all other atoms
        # ----------------------------------------------------

        keep_atoms = set(
            [C2_ATOM] + R3_atoms
        )

        excluded_atoms = [
            i for i in range(1, n_atoms + 1)
            if i not in keep_atoms
        ]


        # ----------------------------------------------------
        # Sterimol calculation
        #
        # dummy_index   = C2 (atom 2)
        # attached_index = first R3 atom (atom 28)
        # ----------------------------------------------------

        sterimol = Sterimol(
            elements,
            coordinates,
            dummy_index=C2_ATOM,
            attached_index=R3_START_ATOM,
            excluded_atoms=excluded_atoms,
            radii_type="crc",
            n_rot_vectors=3600
        )


        # ----------------------------------------------------
        # R3 name
        # ----------------------------------------------------

        R3 = R3_names.get(
            file,
            os.path.splitext(file)[0]
        )


        # ----------------------------------------------------
        # Save results
        # ----------------------------------------------------

        results.append({

            "File": file,

            "R3": R3,

            "C2 atom": C2_ATOM,

            "R3 start atom": R3_START_ATOM,

            "R3 atoms":
                f"{R3_START_ATOM}-{n_atoms}",

            "B1 (Å)":
                sterimol.B_1_value,

            "B5 (Å)":
                sterimol.B_5_value,

            "L (Å)":
                sterimol.L_value

        })


        print(
            f"Processed: {file:25s} "
            f"R3={R3:15s} "
            f"B1={sterimol.B_1_value:.2f} Å  "
            f"B5={sterimol.B_5_value:.2f} Å  "
            f"L={sterimol.L_value:.2f} Å"
        )


    except Exception as e:

        print(f"ERROR in {file}: {e}")


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(results)


# ============================================================
# SAVE CSV
# ============================================================

csv_file = os.path.join(
    xyz_folder,
    "R3_Sterimol_results.csv"
)

df.to_csv(
    csv_file,
    index=False
)


# ============================================================
# SAVE EXCEL
# ============================================================

xlsx_file = os.path.join(
    xyz_folder,
    "R3_Sterimol_results.xlsx"
)

df.to_excel(
    xlsx_file,
    index=False
)


# ============================================================
# DISPLAY
# ============================================================

print("\n==========================================")
print("STERIMOL CALCULATION COMPLETED")
print("==========================================")

print("CSV   :", csv_file)
print("Excel :", xlsx_file)

display(df)

Mounted at /content/drive
Processed: RR_CH2-Ph.xyz             R3=RR_CH2-Ph       B1=1.70 Å  B5=5.85 Å  L=6.87 Å
Processed: RR_CH2-dioxane.xyz        R3=RR_CH2-dioxane  B1=1.70 Å  B5=5.25 Å  L=6.14 Å
Processed: RR_CH2CN.xyz              R3=RR_CH2CN        B1=1.70 Å  B5=3.99 Å  L=4.44 Å
Processed: RR_CH2CO2Me.xyz           R3=RR_CH2CO2Me     B1=1.70 Å  B5=4.96 Å  L=5.78 Å
Processed: RR_CMe2CO2Me.xyz          R3=RR_CMe2CO2Me    B1=2.84 Å  B5=5.07 Å  L=5.51 Å
Processed: RR_Cy.xyz                 R3=RR_Cy           B1=2.01 Å  B5=3.68 Å  L=6.75 Å
Processed: RR_Et.xyz                 R3=RR_Et           B1=1.70 Å  B5=3.26 Å  L=4.68 Å
Processed: RR_Me-Propane.xyz         R3=RR_Me-Propane   B1=1.70 Å  B5=4.50 Å  L=5.80 Å
Processed: RR_Me.xyz                 R3=RR_Me           B1=1.70 Å  B5=2.13 Å  L=3.63 Å
Processed: RR_Ph.xyz                 R3=RR_Ph           B1=1.70 Å  B5=3.27 Å  L=6.90 Å
Processed: RR_alkene.xyz             R3=alkene          B1=1.70 Å  B5=3.18 Å  L=4.83 Å
Processed: RR_alk

,File,R3,C2 atom,R3 start atom,R3 atoms,B1 (Å),B5 (Å),L (Å)
0,RR_CH2-Ph.xyz,RR_CH2-Ph,18,44,44-57,1.700000,5.852931,6.869516
1,RR_CH2-dioxane.xyz,RR_CH2-dioxane,18,44,44-56,1.700000,5.245980,6.141713
2,RR_CH2CN.xyz,RR_CH2CN,18,44,44-48,1.700000,3.992223,4.438541
3,RR_CH2CO2Me.xyz,RR_CH2CO2Me,18,44,44-53,1.700000,4.957261,5.777883
4,RR_CMe2CO2Me.xyz,RR_CMe2CO2Me,18,44,44-59,2.841477,5.068311,5.512985
5,RR_Cy.xyz,RR_Cy,18,44,44-60,2.006980,3.676088,6.749453
6,RR_Et.xyz,RR_Et,18,44,44-50,1.700000,3.255715,4.676969
7,RR_Me-Propane.xyz,RR_Me-Propane,18,44,44-56,1.700000,4.502030,5.803826
8,RR_Me.xyz,RR_Me,18,44,44-47,1.700000,2.129183,3.625097
9,RR_Ph.xyz,RR_Ph,18,44,44-54,1.700000,3.272252,6.899519
